In [1]:
import matplotlib
import matplotlib.pyplot as plt
import backtrader as bt
import pandas as pd

In [2]:
class MyBuySell(bt.observers.BuySell):
    plotlines = dict(
        buy=dict(marker='^', markersize=8.0, color='blue', fillstyle='full'),
        sell=dict(marker='v', markersize=8.0, color='red', fillstyle='full')
    )

In [3]:
class SmaStrategy(bt.Strategy):
    params = (('ma_period', 20), )

    def __init__(self):
        
        #종가(close) 추적
        self.data_close = self.datas[0].close

         #주문(order)/주가(price)/수수료(comm) 추적
        self.order = None
        self.price = None
        self.comm = None

        #분석기간
        print("단순 이동 평균(SMA) 분석기간 =", self.params.ma_period)

        #단순 이동 평균(SMA) 추적
        self.sma = bt.ind.SMA(self.datas[0], period=self.params.ma_period)

    def log(self, txt):
        '''Logging function'''
        dt = self.datas[0].datetime.date(0).isoformat()
        print(f'{dt}, {txt}')

    # 주문(Order)의 상태가 바뀌었을 때 호출하는 함수
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]: #Submitted(주문제출), Accepted(주문접수)는 실제로 거래가 이루어진 것이 아니기에 return
            return

        if order.status in [order.Completed]:                 #Completed(거래체결)
            if order.isbuy():
                self.log(f'매수 주문 완료 --- 주문가(Price): {order.executed.price:.2f}, 거래가(Cost): {order.executed.value:.2f}, 수수료(Commission): {order.executed.comm:.2f}')
                self.price = order.executed.price
                self.comm = order.executed.comm
            else:
                self.log(f'매도 주문 완료 --- 주문가(Price): {order.executed.price:.2f}, 거래가(Cost): {order.executed.value:.2f}, 수수료(Commission): {order.executed.comm:.2f}')

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:  #Canceled(주문취소), Margin(자금부족), Rejected(주문거부)
            self.log(f'주문 실패 확인 --- 사유: {order.getstatusname()}')

        self.order = None
    
    #거래(Trade)가 끝났을 때 그 거래의 손익을 알려주는 함수
    def notify_trade(self, trade):
        if not trade.isclosed:
            return

        self.log(f'거래 결과 확인 --- 거래수익(Gross): {trade.pnl:.2f}, 순수익(Net): {trade.pnlcomm:.2f}')

    #매매 판단 및 주문을 수행
    def next(self):

        if self.order:
            return

        if not self.position:

            if self.data_close[0] > self.sma[0]:
                self.log(f'매수 주문 생성 --- 매수가(Price): {self.data_close[0]:.2f}')
                self.order = self.buy()
        else:
    
            if self.data_close[0] < self.sma[0]:
                self.log(f'매도 주문 생성 --- 매도가(Price): {self.data_close[0]:.2f}')
                self.order = self.sell()

### 백테스트용 데이터 불러오기(애플 2018년도)

In [4]:
dir_nm = "dailyStock"
target = "AAPL"
file_path = f"{dir_nm}/{target}.csv"

aapl_df = pd.read_csv(file_path, encoding="utf-8")
aapl_df['Date'] = pd.to_datetime(aapl_df['Date'])
aapl_df = aapl_df.set_index("Date")

aapl_df = aapl_df.loc["2018-01-01":"2018-12-31"]

data = bt.feeds.PandasData(dataname=aapl_df)

### 백테스트 설정

In [5]:
cerebro = bt.Cerebro(stdstats = False)

cerebro.adddata(data)
cerebro.broker.setcash(1000.0)
cerebro.addstrategy(SmaStrategy)
cerebro.addobserver(MyBuySell)
cerebro.addobserver(bt.observers.Value)

### 백테스트 실행

In [6]:
print(f'포트폴리오 시작가: {cerebro.broker.getvalue():.2f}')
cerebro.run()
print(f'포트폴리오 종료가: {cerebro.broker.getvalue():.2f}')

포트폴리오 시작가: 1000.00
단순 이동 평균(SMA) 분석기간 = 20
2018-02-14, 매수 주문 생성 --- 매수가(Price): 41.84
2018-02-15, 매수 주문 완료 --- 주문가(Price): 42.45, 거래가(Cost): 42.45, 수수료(Commission): 0.00
2018-03-19, 매도 주문 생성 --- 매도가(Price): 43.83
2018-03-20, 매도 주문 완료 --- 주문가(Price): 43.81, 거래가(Cost): 42.45, 수수료(Commission): 0.00
2018-03-20, 거래 결과 확인 --- 거래수익(Gross): 1.36, 순수익(Net): 1.36
2018-04-10, 매수 주문 생성 --- 매수가(Price): 43.31
2018-04-11, 매수 주문 완료 --- 주문가(Price): 43.06, 거래가(Cost): 43.06, 수수료(Commission): 0.00
2018-04-20, 매도 주문 생성 --- 매도가(Price): 41.43
2018-04-23, 매도 주문 완료 --- 주문가(Price): 41.71, 거래가(Cost): 43.06, 수수료(Commission): 0.00
2018-04-23, 거래 결과 확인 --- 거래수익(Gross): -1.35, 순수익(Net): -1.35
2018-05-02, 매수 주문 생성 --- 매수가(Price): 44.14
2018-05-03, 매수 주문 완료 --- 주문가(Price): 43.97, 거래가(Cost): 43.97, 수수료(Commission): 0.00
2018-06-15, 매도 주문 생성 --- 매도가(Price): 47.21
2018-06-18, 매도 주문 완료 --- 주문가(Price): 46.97, 거래가(Cost): 43.97, 수수료(Commission): 0.00
2018-06-18, 거래 결과 확인 --- 거래수익(Gross): 3.00, 순수익(Net): 3.00
2018-07-06, 매수 주

### 백테스트 결과 도식화

In [8]:
figs = cerebro.plot(
    iplot=False,
    volume=False)